# Course notebook — Reasoning and Tool Use

Understand when inference-time search, planning, verification, and external tools improve task outcomes.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · A permit desk with three sources of truth

Ravi wants to help permit clerks answer requests that span an address register, planning rules, and human approval. Some checks are deterministic. Some require evidence from more than one source. The clerk remains responsible for sending the final decision.

The workplace scenario is fictional.


## Search and planning


### Lecture 01 · Reasoning as Search

Separate recall from deliberate search and study decomposition, candidate generation, backtracking, and stopping.

#### Search only where choices change the outcome

One address has two parcel candidates. Another has exactly one registered parcel. Treating both as a large search slows the common case.

Use deterministic lookup for unique matches. Reserve a bounded candidate search for ambiguous inputs and score candidates using evidence the clerk can independently inspect.

#### Read the mechanism

**decomposition**

Decomposition breaks complex tasks into subtasks with clearer stopping criteria and verifiable intermediate results. It enables parallel work and localized retries.

Research splits into search, read, synthesize, and cite—each subtask has its own eval.

**search**

Search explores a space of partial solutions—plans, code candidates, tool sequences—guided by heuristics and budgets. Inference-time search trades compute for accuracy.

Tree-of-thought explores multiple math solution paths before committing to an answer.

**backtracking**

Backtracking abandons partial solutions that fail constraints and returns to earlier choices. Essential when early greedy decisions lock in errors.

If tool call returns 404, backtrack to alternate query formulation instead of hallucinating data.

**heuristics**

Heuristics estimate remaining cost or promise of partial solutions to guide search toward promising branches. Good heuristics cut compute; bad ones waste it or break optimality guarantees.

Manhattan distance guides grid navigation; an overestimated heuristic can make A* suboptimal or incomplete.

**termination**

Termination criteria stop search, agent loops, or generation when goals are met, budgets exhausted, or progress stalls. Without them, systems loop indefinitely.

Stop after five tool calls, success, or three consecutive no-progress iterations.

#### Worked example

**Situation:** A research workflow must plan, call tools, and reject unsupported conclusions when answering whether a policy change affects remote workers in two countries.

**Baseline:** Single-shot model answer from parametric memory—confident but unsourced.

**Application:** Decompose question into sub-queries, search policy state space with explicit backtracking when evidence conflicts, terminate when support threshold met or budget exhausted.

**Test cases:** (1) Normal: both countries covered in one doc. (2) Boundary: evidence only for one country. (3) Adversarial: contradictory paragraphs requiring branch exploration.

**Measurement:** Answer accuracy vs search nodes expanded; unsupported claim rate.

**Design question:** What stopping rule prevents infinite refinement loops on ambiguous policy text?

**Predict before running.** Additional inference helps when the task benefits from exploring and rejecting alternatives. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.1: Reasoning as Search"""

states = [("start", ["search US", "search CA"]), ("search US", ["merge"]), ("search CA", ["merge"])]
budget = 3
expanded = 0
agenda = ["start"]
while agenda and expanded < budget:
    node = agenda.pop(0)
    expanded += 1
    next_nodes = dict(states).get(node, [])
    agenda.extend(next_nodes)
print({"expanded": expanded, "remaining": agenda})


#### Interpret and apply

Solve a constraint problem with explicit state search.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. When does additional inference help via exploring alternatives?
2. How does backtracking appear in policy research tasks?
3. What baseline answers without search?

<details><summary>Check your reasoning</summary>

Q1: When tasks need rejecting wrong branches, not just first guess. Q2: Conflict between paragraphs triggers alternate path. Q3: Zero-shot answer with no tool or retrieval loop.

</details>


### Lecture 02 · Planning

Represent goals, prerequisites, steps, dependencies, state, uncertainty, and replanning without confusing a plausible plan with execution.

#### The order of checks protects the counter staff

A plan starts by drafting approval and then checks whether the parcel is even in the jurisdiction. A polished draft can bias the subsequent review.

Make eligibility a dependency before drafting. Represent missing evidence as a reason to stop and test the plan with a parcel outside the jurisdiction so the workflow never creates a misleading approval draft.

#### Read the mechanism

**goal decomposition**

Goal decomposition maps a top-level objective into subgoals with success conditions and dependencies. It clarifies what 'done' means at each level.

'Ship feature' decomposes into spec approved, code merged, eval passed, and canary clean.

**plan representation**

Plan representation encodes steps, preconditions, effects, and dependencies in structures machines can validate—DAGs, STRIPS, or typed JSON plans.

A migration plan lists DB schema change before app deploy as a hard dependency edge.

**dependencies**

Dependencies constrain execution order—step B requires output or state from step A. Violating them causes flaky failures or data corruption.

Sending customer emails before database migration commits references wrong product IDs.

**replanning**

Replanning updates the action sequence when observations invalidate assumptions. Static plans fail in open environments with changing data.

If inventory check shows zero stock, replan from 'ship item' to 'notify backorder'.

**state**

State captures variables the system believes true at a point in execution—inventory, user intent, pending approvals. Explicit state enables recovery and verification.

Agent state tracks current_step, artifacts_created, and budget_remaining across turns.

#### Worked example

**Situation:** The research workflow must produce a validated plan: gather sources, compare jurisdictions, draft summary—steps have dependencies and prerequisites.

**Baseline:** Model outputs numbered list with hidden dependency violations.

**Application:** Build planner emitting DAG of steps with prerequisites; validate acyclicity and required tools; replan when observation shows missing document.

**Test cases:** (1) Normal: linear plan with clear deps. (2) Boundary: parallelizable searches. (3) Adversarial: circular dependency "approve before fetch."

**Measurement:** Plan validity rate, replans per task, wall-clock vs ad-hoc prompting.

**Design question:** What observation should trigger replanning rather than continuing the current branch?

**Predict before running.** Plans are hypotheses about action sequences and must be updated by observations. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.2: Planning"""

steps = {"fetch_US": [], "fetch_CA": [], "compare": ["fetch_US", "fetch_CA"], "draft": ["compare"]}
def valid_plan(completed):
    for step, prereqs in steps.items():
        if step in completed and not all(p in completed for p in prereqs):
            return False, step
    return True, "ok"
completed = {"fetch_US", "compare"}
print(valid_plan(completed))


#### Interpret and apply

Build a planner that outputs a validated dependency graph.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are plans hypotheses rather than guarantees?
2. How do dependency graphs prevent impossible tool order?
3. What baseline lists steps without validation?

<details><summary>Check your reasoning</summary>

Q1: Execution reveals missing tools, data, or permissions. Q2: Validator blocks compare before both fetches complete. Q3: Free-form bullet plan from single prompt.

</details>


**Section checkpoint.** Specify the goal, allowed actions, and stop condition for a small research workflow.


## Verification and tool contracts


### Lecture 03 · Verification and Critique

Use deterministic checks, tests, rubrics, critics, self-consistency, best-of-N, and external evidence.

#### A second pass needs independent evidence

A model proposes that a permit requirement has been met. Asking the same model if it agrees produces agreement even when the official record is missing.

Verify against a rule or a separate evidence source. Keep proposal and verification outcomes separate, and make one test where fluent wording accompanies missing proof.

#### Read the mechanism

**verifiers**

Verifiers check candidate outputs with independent logic—unit tests, schemas, calculators—not the same model that generated them.

A Python assert verifies JSON plan steps include all required migration phases.

**critique**

Critique models or rubrics evaluate drafts and suggest fixes before finalization. Separating generation from critique reduces shared blind spots.

A critic flags unsupported claims in a research draft before user delivery.

**self-consistency**

Self-consistency samples multiple reasoning paths and aggregates answers by majority vote. It improves reliability when individual samples are noisy.

Five chain-of-thought samples that agree on '42' outweigh one outlier '41'.

**best-of-N**

Best-of-N generates N candidates and selects the best by a scorer or verifier. Quality rises with N but so do cost and latency.

Generate ten JSON plans; pick the one passing all schema and dependency checks.

**tests**

Tests provide executable specifications for tools, plans, and outputs in reasoning pipelines. They turn vague correctness into pass/fail signals.

A migration plan test asserts rollback step exists before destructive changes.

#### Worked example

**Situation:** The workflow generates three draft answers; one sounds best but cites nonexistent sections.

**Baseline:** Pick the most fluent candidate by model self-rank.

**Application:** Generate N candidates, score with independent verifier (citation overlap, unit tests on claims, rubric checklist), select best-of-N, reject all if none pass threshold.

**Test cases:** (1) Normal: one candidate fully verified. (2) Boundary: tie scores within noise. (3) Adversarial: fluent answer failing citation check.

**Measurement:** Verifier precision, gain over single-sample, extra latency/token cost.

**Design question:** Why must verification use signals different from generation?

**Predict before running.** Verification should exploit signals different from those used to generate the answer. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.3: Verification and Critique"""

candidates = [
    {"text": "Cap is 240", "cite_ok": True, "score": 0.7},
    {"text": "Cap is 300", "cite_ok": False, "score": 0.9},
]
def select(cands):
    passing = [c for c in cands if c["cite_ok"]]
    return max(passing, key=lambda c: c["score"]) if passing else None
print("selected:", select(candidates))


#### Interpret and apply

Generate several candidates and select with an independent verifier.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why should verification differ from generation signals?
2. When does self-consistency fail?
3. What baseline picks first candidate?

<details><summary>Check your reasoning</summary>

Q1: Generator optimizes fluency; verifier checks external criteria. Q2: All samples share same hallucination mode. Q3: Single pass with no independent checks.

</details>


### Lecture 04 · Tools as Capability Boundaries

Design typed tools, schemas, descriptions, errors, timeouts, idempotency, permissions, and audit records.

#### Read access and write authority are different grants

The clerk’s assistant should be able to retrieve a parcel record but only a human reviewer can file the completed decision.

Make read and write separate tools with independent authorizations. Confirm the caller and target record at the service boundary. Verify the behavior with clerks who can read but lack filing authority.

#### Read the mechanism

**function calling**

Function calling lets models emit structured invocations with typed arguments that runtime code validates and executes.

Searching internal docs via a read-only tool returns live titles instead of hallucinated links.

**tool schemas**

Tool schemas define parameter names, types, required fields, and descriptions models use to construct calls. Ambiguous schemas cause systematic argument errors.

date_iso string format in schema prevents models passing 'next Tuesday' unparseably.

**idempotency**

Idempotent tools produce the same effect when called repeatedly with the same idempotency key. Agents retry safely only when tools support this.

create_ticket with idempotency key 'abc' must not spawn duplicate tickets on retry.

**timeouts**

Timeouts cap how long tools or model calls may run before cancellation. They prevent hung workflows from blocking resources indefinitely.

A 30-second web search timeout returns partial results instead of freezing the agent.

**permissions**

Permissions bind tools and data access to authenticated identities and roles. Models must not bypass authorization by guessing URLs or parameters.

delete_user tool requires admin role verified server-side, not in the prompt.

#### Worked example

**Situation:** The research assistant calls external APIs; probabilistic tool arguments must not cause unauthorized writes.

**Baseline:** Pass raw model JSON directly to HTTP client.

**Application:** Wrap read-only search API as typed tool with schema, timeouts, idempotency keys, permission checks, structured errors returned to model.

**Test cases:** (1) Normal: valid query string. (2) Boundary: empty query rejected. (3) Adversarial: fuzz malformed types and oversized payloads.

**Measurement:** Schema rejection rate, timeout compliance, zero unauthorized mutations in red-team set.

**Design question:** Where exactly does probabilistic intent cross into deterministic execution?

**Predict before running.** Probabilistic intent must cross a deterministic, authorized boundary before effects occur. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.4: Tools as Capability Boundaries"""

def search_tool(query: str) -> dict:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query required")
    if len(query) > 200:
        raise ValueError("query too long")
    return {"results": [f"hit for {query!r}"]}
for q in ["budget policy", "", 123]:
    try:
        print(search_tool(q) if isinstance(q, str) else search_tool(str(q)))
    except ValueError as e:
        print({"error": str(e)})


#### Interpret and apply

Wrap a read-only API as a typed tool and fuzz its arguments.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are typed tool boundaries required?
2. How do timeouts protect the agent loop?
3. What baseline calls APIs with unvalidated model output?

<details><summary>Check your reasoning</summary>

Q1: Effects need authorization and schema enforcement. Q2: Hung tools exhaust step budget. Q3: Direct exec of model-produced shell/HTTP.

</details>


**Section checkpoint.** Pass a complete plan and then swap two dependent steps. Compare the verifier's decisions.


## Integrations and reasoning cost


### Lecture 05 · MCP and Integration Protocols

Understand clients, servers, tools, resources, prompts, discovery, transport, authentication, and protocol security.

#### A discovered tool is still a dependency

A connector’s lookup contract changes and starts returning a nullable address. The workflow used to assume the field was always populated.

Keep a compatibility check at the tool boundary and treat missing data explicitly. A protocol simplifies discovering capabilities, while the application still owns compatibility, permitted usage, and the fallback.

#### Read the mechanism

**MCP**

Model Context Protocol standardizes how clients discover tools, resources, and prompts from servers. It reduces bespoke integration code but not trust decisions.

An MCP server exposes filesystem read tools; the client still enforces path allowlists.

**resources**

MCP resources expose readable data—files, records, configs—to clients with URI identifiers. Resource access must respect same authorization as APIs.

resource://policy/2024 exposes the PDF bytes; listing must not leak unauthorized URIs.

**tool discovery**

Tool discovery lets clients list available tools and schemas at runtime instead of hardcoding integrations. Discovery responses must be filtered by permission.

A client sees only search_docs, not admin_delete, when connected with read-only scope.

**transports**

MCP transports—stdio, SSE, HTTP—carry protocol messages between clients and servers. Choice affects latency, deployment, and security boundaries.

Stdio suits local IDE agents; SSE suits remote servers behind auth proxies.

**authentication**

Authentication verifies identity of users, clients, and services before access to models, tools, or data. It applies equally to MCP sessions, enterprise assistants, and REST APIs.

OAuth tokens gate MCP server access; SSO identifies employees before internal doc retrieval.

#### Worked example

**Situation:** Internal tools expose HR policies via MCP; a hostile client attempts discovery of admin-only resources.

**Baseline:** Trust any connected client equally.

**Application:** Implement local MCP server exposing read tools, authenticate clients, validate hostile list-resources requests, log transport errors, deny escalation paths.

**Test cases:** (1) Normal: authorized client lists tools. (2) Boundary: expired token. (3) Adversarial: client requests resource outside declared scope.

**Measurement:** Unauthorized access attempts blocked, discovery latency, audit log completeness.

**Design question:** What does MCP standardize—and what must your org still decide?

**Predict before running.** Protocols standardize capability exchange; they do not remove authorization or trust decisions. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.5: MCP and Integration Protocols"""

CHAPTER = "7.5"
print("chapter hook:", CHAPTER)
CLIENT_SCOPES = {"analyst": ["search_policy"]}
REQUEST = {"client": "analyst", "tool": "admin_delete"}
def authorize(client, tool):
    return tool in CLIENT_SCOPES.get(client, [])
print({"allowed": authorize(REQUEST["client"], REQUEST["tool"])})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Implement a small local MCP server and test a hostile client request.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What does MCP standardize versus leave to implementers?
2. How is tool discovery different from authorization?
3. What integration baseline has no auth on local tools?

<details><summary>Check your reasoning</summary>

Q1: Capability advertisement and transport—not trust decisions. Q2: Discovery lists possibilities; auth gates each call. Q3: Open localhost port executing any JSON command.

</details>


### Lecture 06 · Reasoning-System Economics

Balance accuracy, latency, token use, parallel candidates, tool calls, caches, failure rates, and task value.

#### A second attempt should buy evidence

The first plan cannot disambiguate two parcel candidates. Ravi can pay for another reasoning pass or ask the clerk for one missing field.

Compare expected resolution time and review cost. A second call is justified if it can use new evidence or recover from a transient failure. If the address register lacks a field, a human clarification is more useful.

#### Read the mechanism

**test-time compute**

Test-time compute spends extra inference—search, sampling, verification—at query time to improve accuracy. It trades latency and cost for quality on hard inputs.

Spending 5× tokens on best-of-N may be worth it for $10k loan decisions only.

**latency**

Latency is time from request to usable response—dominated by model, retrieval, tools, and serialization. User workflows break when p95 exceeds interaction tolerance.

Adding reranking adds 200ms; measure whether task success gain justifies it.

**cost-quality curves**

Cost-quality curves plot spend—tokens, GPU seconds, API dollars—against task metrics. They guide routing and when to stop adding compute.

Best-of-N may lift accuracy 2 points for 4× cost—acceptable only above a revenue threshold.

**routing**

Routing directs requests to models, tools, or strategies by task type, risk, or budget. Routers encode product policy about cheap versus capable paths.

Simple FAQs route to small model; compliance questions route to audited large model.

**budgets**

Budgets cap tokens, tool calls, wall time, or dollars per task or session. Hard budgets prevent runaway agents and make economics predictable.

A research agent stops after $0.50 API spend or ten tool calls, whichever comes first.

#### Worked example

**Situation:** Leadership wants higher answer quality but budget caps tokens and tool calls per research task.

**Baseline:** Always run best-of-5 with full verifier loop—quality up, costs unsustainable.

**Application:** Plot cost-quality curves for single-pass, best-of-N, verifier loops; route easy queries cheaply, spend test-time compute only on high-value uncertain cases.

**Test cases:** (1) Normal: low-uncertainty FAQ. (2) Boundary: uncertainty score near routing threshold. (3) Adversarial: attacker triggers expensive loops via ambiguous queries.

**Measurement:** Quality by route tier, average $/task, loop explosion incidents.

**Design question:** What signal routes a query to expensive reasoning without sending everything there?

**Predict before running.** Spend additional computation only where expected outcome improvement justifies it. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 7.6: Reasoning-System Economics"""

routes = [
    {"name": "single", "cost": 1, "quality": 0.78},
    {"name": "best3", "cost": 3, "quality": 0.86},
    {"name": "verify", "cost": 5, "quality": 0.91},
]
def pick(uncertainty, budget):
    opts = [r for r in routes if r["cost"] <= budget]
    if uncertainty < 0.3:
        return opts[0]
    return max(opts, key=lambda r: r["quality"])
print(pick(0.25, 4))
print(pick(0.8, 4))


#### Interpret and apply

Plot quality and cost for single-pass, best-of-N, and verifier loops.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. When is extra test-time compute worth the cost?
2. How do attackers exploit unbounded reasoning loops?
3. What baseline always maximizes quality?

<details><summary>Check your reasoning</summary>

Q1: When expected value of correctness gain exceeds marginal cost on slice. Q2: Ambiguity triggers repeated tool/model cycles. Q3: Best-of-N plus verifier on every query.

</details>


**Section checkpoint.** List the minimum tool fields and estimate the budget for one verification retry.


## Course workshop

Construct a plan that contains every step but places the final review too early. Compare it with the valid plan.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Generate candidate plans, verify constraints, and select the cheapest valid plan."""
PLANS = [
    ["read", "publish"],
    ["read", "draft", "review", "publish"],
    ["draft", "read", "publish"],
]


def verify(plan):
    required = {"read", "draft", "review", "publish"}
    if not required.issubset(plan): return False, "missing required step"
    positions = {step: plan.index(step) for step in required}
    if not positions["read"] < positions["draft"] < positions["review"] < positions["publish"]:
        return False, "invalid dependency order"
    return True, "valid"


valid = []
for plan in PLANS:
    ok, reason = verify(plan); print(plan, ok, reason)
    if ok: valid.append(plan)
print("selected=", min(valid, key=len))


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert verify(PLANS[1])[0]
assert not verify(list(reversed(PLANS[1])))[0]
print('Workshop checks passed')


## Course project

Build a research workflow with a planner, typed tools, evidence store, verifier, and bounded recovery.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
